Cell 1: Load API Key & PDF Chunks (Recap of Ingestion)

In [1]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load GOOGLE_API_KEY from .env file
load_dotenv()

# 1. Load the PDF page by page
loader = PyPDFLoader("data/attention.pdf")
docs = loader.load()

# 2. Split pages into overlapping chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(docs)

print(f"Total chunks ready for embedding: {len(chunks)}")

C:\Users\Harshitha\AppData\Local\Temp\ipykernel_20840\967960741.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Total chunks ready for embedding: 52


Cell 2: Create Embeddings & Store in Vector DB (FAISS)

In [2]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

# 1. Initialize the Gemini embedding model (converts text -> vectors)
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# 2. Convert all chunks into vectors and store them in FAISS Vector DB
vector_db = FAISS.from_documents(chunks, embeddings)

# 3. Save the Vector DB locally on disk so you don't re-embed every time
vector_db.save_local("faiss_index")
print("Vector DB created and saved to 'faiss_index' folder!")

Vector DB created and saved to 'faiss_index' folder!


Cell 3: Query Retrieval (Similarity Search & Retriever)

In [3]:
# Load the saved Vector DB back from disk (optional, shows how persistence works)
loaded_db = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True  # Required by FAISS when loading local pickle files
)

# Method A: Direct similarity search (returns top 3 matching chunks)
query = "What is Scaled Dot-Product Attention?"
matched_docs = loaded_db.similarity_search(query, k=3)

print("Top matching chunk preview:\n", matched_docs[0].page_content[:300])

# Method B: Convert Vector DB into a Retriever interface (used inside chains)
retriever = loaded_db.as_retriever(search_kwargs={"k": 3})

Top matching chunk preview:
 Scaled Dot-Product Attention
 Multi-Head Attention
Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several
attention layers running in parallel.
of the values, where the weight assigned to each value is computed by a compatibility function of the
query with th


Cell 4: Complete End-to-End RAG Chain (Retriever + Prompt + Gemini LLM)

In [4]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Load a less-crowded Gemini model and tell it to auto-retry 5 times if servers get busy
llm = init_chat_model("google_genai:gemini-3.5-flash-lite", max_retries=5)

# 2. Create the instruction template so the model only uses facts from your PDF
prompt = ChatPromptTemplate.from_template("""
Answer the user's question using ONLY the context provided below.
If the answer is not in the context, say "I don't know based on the provided document."

Context:
{context}

Question:
{question}
""")

# 3. Helper function to combine the 3 retrieved PDF chunks into one plain text block
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 4. Connect all steps together from left to right using the pipe (|) symbol:
#    - retriever finds the top 3 matching chunks and format_docs joins them as {context}
#    - RunnablePassthrough() passes your question straight into {question}
#    - prompt puts {context} and {question} together
#    - llm reads the prompt and writes the answer
#    - StrOutputParser() extracts just the clean text answer
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 5. Run the chain with your question and print the answer
response = rag_chain.invoke("Explain the Transformer architecture in simple terms based on this paper.")
print(response)

Based on the provided document, the Transformer architecture uses stacked self-attention and point-wise, fully connected layers for both the encoder and the decoder:

* **Encoder:** It is made up of a stack of $N = 6$ identical layers. Each layer has two sub-layers: a multi-head self-attention mechanism and a simple, position-wise fully connected feed-forward network. A residual connection followed by layer normalization is used around each sub-layer ($\text{LayerNorm}(x + \text{Sublayer}(x))$).
* **Decoder:** It is also made up of a stack of $N = 6$ identical layers. In addition to the two sub-layers found in the encoder, the decoder includes a third sub-layer that performs multi-head attention over the encoder stack's output. It also uses residual connections and layer normalization around each sub-layer, and its self-attention sub-layer is modified with masking to prevent positions from attending to subsequent positions.
* **Feed-Forward Networks:** Each layer in both the encoder an

## End-to-End RAG Pipeline — Step-by-Step Explanation

### Cell 1: Loading the PDF & Splitting It into Chunks
* **`load_dotenv()`**: Reads the `.env` file and loads `GOOGLE_API_KEY` into memory so Google's embedding and Gemini models can authenticate automatically without hardcoding secret keys.
* **`PyPDFLoader("data/attention.pdf").load()`**: Opens the PDF and reads it one page at a time, returning a Python list of `Document` objects (`page_content` + `metadata` containing the file path and page number).
* **`RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)`**: Cuts full pages into smaller chunks of at most 1,000 characters, splitting cleanly at paragraph breaks and spaces instead of cutting words in half. `chunk_overlap=200` repeats the last 200 characters of one chunk at the start of the next chunk so context is not lost across split boundaries.

---

### Cell 2: Creating Embeddings & Storing Them in a Vector Database
* **`GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")`**: Loads Google's embedding model, which converts text strings into lists of decimal numbers (vectors) that capture semantic meaning.
* **`FAISS.from_documents(chunks, embeddings)`**: Converts every text chunk into a numerical vector and stores all vectors alongside their original text and metadata inside a local FAISS vector database.
* **`vector_store.save_local("faiss_index")`**: Saves the entire vector database to the `faiss_index/` folder on disk so the PDF does not need to be re-parsed or re-embedded on future runs.

---

### Cell 3: Searching the Vector Database (Similarity Search & Retriever)
* **`vector_store.similarity_search(query, k=3)`**: Converts the `query` string into a vector using the same embedding model, compares it mathematically against all stored chunk vectors, and returns the top 3 (`k=3`) closest matching `Document` chunks.
* **`vector_store.as_retriever(search_kwargs={"k": 3})`**: Wraps the FAISS vector store into a standard LangChain Retriever that takes a question string as input and outputs the top 3 matching `Document` objects.

---

### Cell 4: Building the Complete RAG Chain (`rag_chain`)
* **`init_chat_model("google_genai:gemini-3.8-flash")`**: Initializes the Gemini chat model that reads the retrieved PDF chunks and writes the final answer.
* **`ChatPromptTemplate.from_template(...)`**: Defines a prompt with `{context}` and `{question}` placeholders, instructing the LLM to answer using **only** the retrieved context so it does not guess or hallucinate.
* **`format_docs(docs)`**: Extracts `.page_content` from each of the 3 retrieved `Document` objects and joins them into one plain text string for `{context}`.
* **LCEL Pipe Chain (`|`)**: Connects every step from left to right:
  1. **`{"context": retriever | format_docs, "question": RunnablePassthrough()}`**: Runs in parallel—`retriever | format_docs` fetches and formats the top 3 chunks into `{context}`, while `RunnablePassthrough()` passes the user's original question directly into `{question}`.
  2. **`| prompt`**: Fills `{context}` and `{question}` into the prompt template.
  3. **`| llm`**: Sends the completed prompt to Gemini to generate an `AIMessage` response.
  4. **`| StrOutputParser()`**: Extracts just the clean plain-text string from Gemini's response.
* **`rag_chain.invoke(...)`**: Runs the entire retrieval and generation pipeline in one call and prints the grounded answer.